In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 7.2 Complex Analysis II: Causality, Kramers–Kronig, Matsubara Sums, and Steepest Descent

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VII — Quantum Statistical Mechanics",
    number="7.2",
    title="Complex Analysis II: Causality, Kramers–Kronig, Matsubara Sums, and Steepest Descent",
    blurb="Analyticity, it turns out, is not optional. Because no effect may precede its "
    "cause, the response functions of matter are analytic in half the complex plane — "
    "and so their absorption determines their dispersion, exactly, by an integral. "
    "Because the thermal weight functions have poles at evenly spaced imaginary "
    "frequencies, the sums of quantum statistics close by contours — and the Bose and "
    "Fermi distributions fall out before we have done any statistical mechanics at "
    "all. And because sharp peaks dominate large-N integrals, Stirling's formula is "
    "four lines. The machinery of §7.1, pointed at physics.",
    difficulty="advanced",
    estimate="190–230 min",
)

## Notebook overview

The previous notebook built the machinery of complex analysis and spent it on integrals chosen for
their beauty. This one spends it on physics, and the difference in register matters: here
analyticity is not a property we *assume* to make calculations tractable — it is a property nature
*forces*, and its consequences are measurable. Four applications carry the argument, each a theorem
of analyticity wearing physical clothing.

First, the **Sokhotski–Plemelj identity**, the grammar in which every propagator and response
function is written: a pole sitting on the integration path is not a disaster but a fork, and the
$i\varepsilon$ prescription that decides how to pass it splits one singular integral into a
principal value (the *dispersive* part of a response) and a delta function (the *absorptive* part).
Second, **causality**: the plain sentence "no effect precedes its cause" makes every causal
response function analytic in the upper half of the complex frequency plane, and analyticity plus
Sokhotski–Plemelj yields the **Kramers–Kronig relations** — absorption at all frequencies
determines dispersion at every frequency, exactly. We verify this to six digits on an old friend,
the damped driven oscillator of Volume I, and check the f-sum rule that real spectroscopies obey.
Third, **summation by residues**: the poles of $\pi\cot(\pi z)$ sit at the integers, so infinite
sums close by contours — the Basel problem falls in three lines — and replacing the cotangent by
its thermal cousins evaluates the **Matsubara sums** of thermal physics, out of which the Bose and
Fermi occupation functions *emerge from contour analysis alone*, before this volume has done any
statistical mechanics. When [§7.7](bose-einstein-fermi-dirac.ipynb) derives the same functions
from the grand canonical ensemble, two
entirely independent routes will have met. Fourth, **steepest descent**: large-$N$ integrals
concentrate onto saddle points, Stirling's formula — borrowed on credit in
[§5.3](../05-classical-stat-mech/large-n-limit.ipynb) — is finally
*derived*, and stationary phase is named as the principle WKB
([§6.23](../06-quantum-mechanics/wkb-semiclassical.ipynb)) was secretly using.

A closing section states **analytic continuation**: the identity theorem (rigidity once more), the
Gamma reflection formula — which the observant reader will recognize as the keyhole integral of
[§7.1](complex-analysis-residues.ipynb) in
disguise — and the Riemann zeta function, including the particular number $\zeta(3/2) = 2.612\dots$
that the Bose–Einstein condensation temperature is made of.

> **Conventions (this notebook).** Fourier transforms of response functions use the physics sign
> $\chi(\omega) = \int_0^\infty \chi(t)\,e^{+i\omega t}\,dt$, so that causality ($\chi(t) = 0$ for
> $t<0$) implies analyticity in the **upper** half-plane, where $e^{i\omega t}$ decays. Real-valued
> response in time implies the reality condition $\chi(-\omega) = \chi(\omega)^*$ (Re $\chi$ even,
> Im $\chi$ odd), which every full-line integral below uses. Every principal value is computed with
> `scipy.integrate.quad(weight='cauchy', wvar=...)` — the weighted rule built for exactly this
> kernel — never by naive quadrature across a pole. Matsubara frequencies are bosonic
> $\omega_n = 2\pi n/\beta$ and fermionic $\omega_n = (2n+1)\pi/\beta$; frequency sums are done by
> symmetric truncation with the cutoff stated and the $1/n^2$ tail estimated, honestly, alongside.
> All large-$N$ factorial comparisons live in log space via `scipy.special.gammaln`, because
> `scipy.special.gamma` overflows beyond $N \approx 170$ — a real float boundary that this volume
> converts into a discipline rather than an accident.
>
> **How to read the checks.** Each exercise closes with a `validate` call against an independent
> fact: the $i\varepsilon$ integrals converging to $\mathrm{P} + i\pi f(x_0)$; the causal poles
> sitting at $\operatorname{Im}\omega = -\gamma/2$ and the numerical Fourier transform of the
> time-domain response reproducing $\chi(\omega)$; the Kramers–Kronig reconstruction landing on the
> exact dispersion to $10^{-6}$ and the f-sum rule giving $\pi/2$; residue-summation closed forms
> against truncated series and `scipy.special.zeta`; Matsubara sums against their $\coth$/$\tanh$
> closed forms to seven digits; Stirling's log-error equalling $1/(12N)$; and the reflection
> formula against `scipy.special.gamma` at real and complex arguments. A ✓ is strong evidence; a ✗
> is a prompt to *locate the discrepancy*.
>
> **Scope.** Four physicist's applications of one-variable complex analysis, computationally.
> Matsubara *Green's functions* — the many-body formalism these frequency sums serve — need second
> quantization and are a Volume VIII horizon, as is the full linear-response/Kubo framework behind
> Kramers–Kronig. See Arfken, Weber & Harris (dispersion relations, saddle-point methods); Jackson,
> *Classical Electrodynamics* (Kramers–Kronig in optics); Mahan, *Many-Particle Physics* or Altland
> & Simons (the Matsubara formalism, for the horizon). Cross-reference
> [§7.1](complex-analysis-residues.ipynb) (the machinery), Volume
> I (the damped driven oscillator),
> [§6.24](../06-quantum-mechanics/time-dependent-perturbation.ipynb) (absorption and linewidths —
> Im $\chi$ is what the golden
> rule computes), [§5.3](../05-classical-stat-mech/large-n-limit.ipynb) (Stirling, now finally
> derived), [§6.23](../06-quantum-mechanics/wkb-semiclassical.ipynb) (WKB),
> [§5.9](../05-classical-stat-mech/grand-canonical-ensemble-equivalence.ipynb) (the ensemble route
> that [§7.7](bose-einstein-fermi-dirac.ipynb)
> will take to the same occupations), and forward to [§7.3](statistical-toolkit.ipynb),
> [§7.7](bose-einstein-fermi-dirac.ipynb), [§7.17](bose-einstein-condensation.ipynb),
> [§7.20](imaginary-time-quantum-classical.ipynb).

## Theory in brief

### Principal values and Sokhotski–Plemelj

An integral whose integrand has a simple pole *on* the path, $\int f(x)/(x - x_0)\,dx$, is given
meaning by the **principal value**: excise a symmetric interval $(x_0-\delta, x_0+\delta)$, let
$\delta \to 0$, and the divergences on the two sides cancel against each other. (Computationally
this is `scipy.integrate.quad(weight='cauchy', wvar=x0)`, which knows the kernel analytically.)
Alternatively, push the pole slightly *off* the path with an $i\varepsilon$ and ask what survives
the limit:

```{math}
:label: eq-sokhotski-plemelj
\lim_{\varepsilon\to 0^+} \frac{1}{x - x_0 \mp i\varepsilon}
= \mathrm{P}\,\frac{1}{x - x_0} \pm i\pi\,\delta(x - x_0) .
```

The derivation is a split into real and imaginary parts: $\frac{1}{x - x_0 - i\varepsilon} =
\frac{x - x_0}{(x-x_0)^2 + \varepsilon^2} + \frac{i\varepsilon}{(x-x_0)^2 + \varepsilon^2}$. The
real part tends to the principal-value kernel; the imaginary part is a Lorentzian of unit area
times $\pi$ — a **nascent delta function**. One pole, two pieces of physics: in a response
function the $\mathrm{P}$ part is the reactive, **dispersive** response and the $\delta$ part is
the dissipative, **absorptive** one. Every $i\varepsilon$ in field theory is this identity.

### Causality forces analyticity

Let $\chi(t)$ be a response function — the output at time $t$ per unit impulse at time $0$ — and
let it be **causal**: $\chi(t) = 0$ for $t < 0$. Its Fourier transform then only integrates over
$t > 0$,

```{math}
:label: eq-causality
\chi(\omega) = \int_0^\infty \chi(t)\,e^{i\omega t}\,dt
\quad\Longrightarrow\quad
\chi \text{ is analytic for } \operatorname{Im}\omega > 0,
```

because for $\operatorname{Im}\omega > 0$ the factor $e^{i\omega t}$ *decays* and the integral
converges — better than converges: it can be differentiated under the integral sign, which is
analyticity. Causality, a statement about time, has become a statement about the complex plane.
Our laboratory specimen is the damped driven oscillator of Volume I, $\ddot x + \gamma\dot x +
\omega_0^2 x = F(t)$, whose susceptibility $\chi(\omega) = 1/(\omega_0^2 - \omega^2 - i\gamma\omega)$
has poles at $\omega = \pm\Omega - i\gamma/2$ with $\Omega = \sqrt{\omega_0^2 - \gamma^2/4}$ —
both in the **lower** half-plane, as causality demands. A real response in time adds the
**reality condition** $\chi(-\omega) = \chi(\omega)^*$: Re $\chi$ is even, Im $\chi$ is odd.

### The Kramers–Kronig relations

Apply the Cauchy integral formula to $\chi$ along the real axis, closed in the analytic upper
half-plane, with a small semicircular detour around the point $\omega$ — Sokhotski–Plemelj
supplies the $\pm i\pi$ boundary term — and analyticity delivers

```{math}
:label: eq-kramers-kronig
\operatorname{Re}\chi(\omega) = \frac{1}{\pi}\,\mathrm{P}\!\int_{-\infty}^{\infty}
\frac{\operatorname{Im}\chi(\omega')}{\omega' - \omega}\,d\omega',
\qquad
\operatorname{Im}\chi(\omega) = -\frac{1}{\pi}\,\mathrm{P}\!\int_{-\infty}^{\infty}
\frac{\operatorname{Re}\chi(\omega')}{\omega' - \omega}\,d\omega' .
```

The physics deserves slow reading: **absorption at all frequencies determines dispersion at every
frequency**, and vice versa — refraction and attenuation are one analytic function seen twice. No
experiment has ever caught a causal medium violating these relations. With the reality condition
folding the integral onto $\omega' > 0$, we verify the first relation to six digits on the
oscillator, and we check the **f-sum rule** $\int_0^\infty \omega \operatorname{Im}\chi(\omega)\,
d\omega = \pi/2$ (for unit mass) — a model-independent constraint tying the whole absorption
spectrum to nothing but inertia, which practical spectroscopy uses as a consistency check on data.
The general linear-response (Kubo) framework behind these relations is Volume VIII territory.

### Summation by residues

The function $\pi\cot(\pi z)$ has a simple pole at every integer $n$, each with residue exactly
$1$. Integrate $f(z)\,\pi\cot(\pi z)$ around a huge contour: if $f$ decays fast enough the
integral vanishes as the contour grows, and the residue theorem converts the sum over integer
poles into (minus) the residues at the poles of $f$ itself,

```{math}
:label: eq-residue-summation
\sum_{n=-\infty}^{\infty} f(n) = -\sum_{\text{poles } z_k \text{ of } f}
\operatorname{Res}_{z=z_k}\big[\pi\cot(\pi z)\,f(z)\big] .
```

The **Basel problem** falls in three lines: $f(z) = 1/z^2$ has its only pole at $0$, where the
Laurent expansion $\pi\cot(\pi z) = 1/z - \pi^2 z/3 - \dots$ gives
$\operatorname{Res}_0[\pi\cot(\pi z)/z^2] = -\pi^2/3$, whence $2\sum_{n\ge1} 1/n^2 = \pi^2/3$ and
$\zeta(2) = \pi^2/6$. The shifted identity $\sum_{n\ge1} 1/(n^2+a^2) = \big(\pi\coth(\pi a)/a -
1/a^2\big)/2$ follows the same way from the poles at $\pm ia$ — and it is the deliberately chosen
warm-up whose thermal cousin is next.

### Matsubara sums: the statistics before the statistics

Thermal physics is full of sums over the **Matsubara frequencies** — bosonic $\omega_n = 2\pi
n/\beta$ and fermionic $\omega_n = (2n+1)\pi/\beta$, with $\beta$ the inverse temperature. These
sums close by contours exactly as the integer sums did, once the cotangent is replaced by its
thermal cousins: the **Bose weight** $\beta\,n_B(z) \equiv \beta/(e^{\beta z}-1)$ has simple poles
with unit residue at precisely $z = i\omega_n$ (bosonic), and $-\beta\,n_F(z) \equiv
-\beta/(e^{\beta z}+1)$ at the fermionic ones. The flagship evaluations,

```{math}
:label: eq-matsubara
\frac{1}{\beta}\sum_{n=-\infty}^{\infty} \frac{1}{\omega_n^2 + \varepsilon^2} =
\begin{cases}
\dfrac{1}{2\varepsilon}\coth\dfrac{\beta\varepsilon}{2}
  = \dfrac{1}{2\varepsilon}\big[1 + 2 n_B(\varepsilon)\big] & \text{(bosonic)}\\[10pt]
\dfrac{1}{2\varepsilon}\tanh\dfrac{\beta\varepsilon}{2}
  = \dfrac{1}{2\varepsilon}\big[1 - 2 n_F(\varepsilon)\big] & \text{(fermionic)},
\end{cases}
```

are verified below to seven digits against symmetric truncations. Pause on what happens in
{eq}`eq-matsubara`: the **Bose and Fermi occupation functions** $n_B(\varepsilon) =
1/(e^{\beta\varepsilon}-1)$ and $n_F(\varepsilon) = 1/(e^{\beta\varepsilon}+1)$ have emerged from
contour integration alone — before this volume has defined an ensemble. When
[§7.7](bose-einstein-fermi-dirac.ipynb) derives the same
functions from the grand canonical machinery of
[§5.9](../05-classical-stat-mech/grand-canonical-ensemble-equivalence.ipynb), two entirely
independent routes will have
met, and their agreement is the kind of consistency physics is built on. (The Matsubara **Green's
functions** these sums serve in many-body theory need second quantization: a named Volume VIII
horizon.)

### Steepest descent and Stirling

Integrals of the form $\int e^{N f(x)}\,dx$ are dominated, for large $N$, by the neighbourhood of
the maximum $x_0$ of $f$: expanding $f$ to second order and doing the resulting Gaussian integral
gives the **Laplace approximation**,

```{math}
:label: eq-steepest-descent
\int e^{N f(x)}\,dx \;\approx\; e^{N f(x_0)}\sqrt{\frac{2\pi}{N\,|f''(x_0)|}} ,
```

with relative corrections in powers of $1/N$. Applied to $N! = \Gamma(N+1) = \int_0^\infty
e^{N\ln t - t}\,dt$ — substitute $t = Ns$ to expose the form, saddle at $s = 1$ — it yields
**Stirling's formula** $N! \approx \sqrt{2\pi N}\,(N/e)^N$ in four lines: the workhorse that
[§5.3](../05-classical-stat-mech/large-n-limit.ipynb)
borrowed on credit is finally derived, and the first correction $1/(12N)$ is visible numerically
at every $N$ we test. The complex generalization deforms the contour through a saddle point along
the path of steepest descent (hence the name); its oscillatory sibling, **stationary phase**, is
the principle the WKB approximation ([§6.23](../06-quantum-mechanics/wkb-semiclassical.ipynb)) was
secretly using, and the one the path integral's
classical limit (the neighbourhood of [§7.20](imaginary-time-quantum-classical.ipynb)) will use.

### Analytic continuation

The **identity theorem** — two functions analytic on a connected domain that agree on any set with
a limit point agree everywhere — is rigidity's sharpest edge: it makes extension beyond a
formula's domain of convergence *unique* whenever it exists at all,

```{math}
:label: eq-continuation
\Gamma(z)\,\Gamma(1-z) = \frac{\pi}{\sin(\pi z)},
\qquad
\zeta(s) = \sum_{n=1}^{\infty}\frac{1}{n^s}\ (\operatorname{Re}s>1)\ \text{, continued to } \mathbb{C}\setminus\{1\} .
```

The reflection formula on the left deserves a double take: the keyhole integral of
[§7.1](complex-analysis-residues.ipynb),
$\int_0^\infty x^{\alpha-1}/(1+x)\,dx = \pi/\sin(\pi\alpha)$, *is* this formula, because the
integral is the Beta function $B(\alpha, 1-\alpha) = \Gamma(\alpha)\Gamma(1-\alpha)$. And the zeta
function's continuation is not a curiosity: $\zeta(2) = \pi^2/6$ closes the Basel circle, while
$\zeta(3/2) = 2.612\dots$ is the number the Bose–Einstein condensation temperature
([§7.17](bose-einstein-condensation.ipynb)) is made
of. We state and check; the systematic theory — the Gamma function's continuation and the zeta
function's functional equation — is carried out in full by Arfken, Weber & Harris,
*Mathematical Methods for Physicists*.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import quad
from scipy.special import gamma as gamma_fn
from scipy.special import gammaln, zeta

from ecp import draw, validate

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"

# Conventions: response-function transforms use e^(+iωt), so causality
# puts analyticity in the UPPER half ω-plane; the reality condition χ(−ω) = conj χ(ω)
# folds full-line integrals onto ω > 0; principal values ALWAYS via
# scipy.integrate.quad(weight='cauchy', wvar=·); Matsubara sums by symmetric truncation
# with the cutoff stated and the 1/n^2 tail estimated; large-N factorials in log space
# (scipy.special.gammaln), never through scipy.special.gamma, which overflows at N ≈ 170.

# The laboratory response function for Exercises 2, 3, and 8: the damped driven
# oscillator of Volume I with unit mass, natural frequency OMEGA0 and damping GAMMA.
OMEGA0, GAMMA = 1.0, 0.3


def chi(w):
    """The damped-oscillator susceptibility χ(ω) = 1/(ω0^2 − ω^2 − iγω) (eq-causality).

    The unit-mass damped driven oscillator of Volume I, x'' + γ x' + ω0^2 x = F(t),
    solved in frequency space with the e^(+iωt) convention. Its two poles sit at
    ω = ±Ω − iγ/2 with Ω = √(ω0^2 − γ^2/4) — both in the LOWER half-plane, which is
    causality's fingerprint and the hypothesis behind Kramers–Kronig. Accepts complex ω
    (Exercise 2 evaluates it off the real axis).

    Parameters
    ----------
    w : float, complex or numpy.ndarray
        Frequency (may be complex).

    Returns
    -------
    complex or numpy.ndarray
        χ(ω), with OMEGA0 and GAMMA from the module scope.
    """
    return 1.0 / (OMEGA0**2 - w**2 - 1j * GAMMA * w)


def principal_value(f, x0, a, b):
    """P ∫_a^b f(x)/(x − x0) dx via scipy.integrate.quad's Cauchy weight.

    The weighted rule (weight='cauchy', wvar=x0) knows the singular kernel 1/(x − x0)
    analytically and integrates only the smooth factor f numerically — the correct tool
    for every principal value in this notebook, and far more accurate than any naive
    symmetric-excision quadrature. The pole must lie strictly inside (a, b).

    Parameters
    ----------
    f : callable
        The smooth numerator f(x).
    x0 : float
        Location of the simple pole, with a < x0 < b.
    a, b : float
        Finite integration limits.

    Returns
    -------
    float
        The principal-value integral.
    """
    return quad(f, a, b, weight="cauchy", wvar=x0)[0]


def kk_real_from_imag(im_chi, w, w_max=500.0):
    """Re χ(ω) from Im χ alone by the Kramers–Kronig relation (eq-kramers-kronig).

    Uses the reality condition (Im χ odd) to fold the full-line integral onto ω' > 0:
    Re χ(ω) = (2/π) P ∫_0^∞ ω' Im χ(ω')/(ω'^2 − ω^2) dω'. The remaining simple pole at
    ω' = ω is handled by writing 1/(ω'^2 − ω^2) = [1/(ω' + ω)]·1/(ω' − ω) and handing
    the second factor to quad's Cauchy weight via :func:`principal_value`. The upper
    limit w_max truncates a tail that decays like γ/ω'^4 for the oscillator — a
    ~5e-10-level bandwidth error at the default, stated rather than hidden (Exercise 8
    studies the finite-bandwidth error deliberately).

    Parameters
    ----------
    im_chi : callable
        The absorptive part Im χ(ω') for ω' ≥ 0.
    w : float
        The frequency at which to reconstruct the dispersive part, 0 < w < w_max.
    w_max : float, optional
        Truncation of the folded integral (default 500.0).

    Returns
    -------
    float
        The reconstructed Re χ(ω).
    """
    g = lambda wp: wp * im_chi(wp) / (wp + w)
    return (2.0 / np.pi) * principal_value(g, w, 0.0, w_max)


def matsubara_sum(g, beta, statistics, n_max):
    """(1/β) Σ_n g(ω_n) over Matsubara frequencies, by symmetric truncation |n| ≤ n_max.

    The frequency grids (eq-matsubara): bosonic ω_n = 2πn/β, fermionic
    ω_n = (2n+1)π/β. For the 1/ω_n^2-decaying summands of this notebook the truncation
    error is a ~Σ_(n>n_max) (β/2πn)^2 tail, i.e. O(1/n_max) — with n_max ~ 3×10^5 that
    is a seventh-digit effect, and Exercise 5 prints the estimate next to the result
    rather than pretending the sum ran to infinity.

    Parameters
    ----------
    g : callable
        Summand as a function of the (real) Matsubara frequency ω_n; vectorized.
    beta : float
        Inverse temperature β.
    statistics : str
        'bose' (ω_n = 2πn/β) or 'fermi' (ω_n = (2n+1)π/β).
    n_max : int
        Symmetric truncation: n runs from −n_max to +n_max.

    Returns
    -------
    float
        The truncated frequency sum divided by β.
    """
    n = np.arange(-n_max, n_max + 1, dtype=float)
    if statistics == "bose":
        w_n = 2.0 * np.pi * n / beta
    elif statistics == "fermi":
        w_n = (2.0 * n + 1.0) * np.pi / beta
    else:
        raise ValueError("statistics must be 'bose' or 'fermi'")
    return float(np.sum(g(w_n)) / beta)


def laplace_approx(f, d2f_x0, x0, N):
    """The Laplace approximation ∫ e^(N f(x)) dx ≈ e^(N f(x0)) √(2π/(N |f''(x0)|)).

    Direct implementation of eq-steepest-descent: the integral concentrates on a
    Gaussian of width ~1/√N around the maximum x0 of f, and the prefactor is that
    Gaussian's area. Returned in LOG form, ln I ≈ N f(x0) + (1/2) ln(2π/(N |f''|)),
    because the notebook's applications (Stirling at N up to 1000) overflow any float
    in linear space — the volume's log-space discipline.

    Parameters
    ----------
    f : callable
        The exponent profile f(x).
    d2f_x0 : float
        The second derivative f''(x0) at the maximum (negative).
    x0 : float
        Location of the interior maximum of f.
    N : float
        The large parameter.

    Returns
    -------
    float
        ln of the Laplace estimate of ∫ e^(N f(x)) dx.
    """
    return N * f(x0) + 0.5 * np.log(2.0 * np.pi / (N * abs(d2f_x0)))

## Exercise 1 — The Sokhotski–Plemelj identity

One pole, two pieces of physics: the $i\varepsilon$ prescription splits a singular integral into a
dispersive principal value and an absorptive delta function, and the split can be watched
numerically. Cite {eq}`eq-sokhotski-plemelj`.

1. Derive the identity by splitting $1/(x - x_0 - i\varepsilon)$ into real and imaginary parts and
   recognizing the nascent delta function.
2. Compute the principal value $\mathrm{P}\int e^{-x^2}/(x - x_0)\,dx$ with
   `scipy.integrate.quad(weight='cauchy', wvar=x0)` (the `principal_value` helper), for
   $x_0 = 0.5$ on $[-8, 8]$.
3. Evaluate the real and imaginary parts of $\int e^{-x^2}/(x - x_0 - i\varepsilon)\,dx$ by plain
   `scipy.integrate.quad` for $\varepsilon = 10^{-1}, 10^{-2}, 10^{-3}$, and confirm convergence
   to the principal value and to $\pi e^{-x_0^2}$ respectively — including the honest,
   first-order-in-$\varepsilon$ rate.
4. Interpret: in a response function, the $\mathrm{P}$ part is the reactive (dispersive) response
   and the $\delta$ part the dissipative (absorptive) one. (Prose part.)

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.check(
    errs_re[1e-3] < 1e-2 and errs_im[1e-3] < 1e-2,
    "Sokhotski–Plemelj: the iε integrals converge to P∫ + iπf(x0)",
    f"at ε=1e-3: Re err {errs_re[1e-3]:.1e}, Im err {errs_im[1e-3]:.1e}",
)
validate.check(
    errs_re[1e-3] < errs_re[1e-1] and errs_im[1e-3] < errs_im[1e-1],
    "the convergence is monotone in ε (first-order rate, honestly first-order)",
)

## Exercise 2 — Causality places the poles

No effect before its cause — and the complex frequency plane keeps the receipt: the poles of a
causal susceptibility have nowhere to live but the lower half-plane. Cite {eq}`eq-causality`.

1. Write the damped-oscillator susceptibility $\chi(\omega) = 1/(\omega_0^2 - \omega^2 -
   i\gamma\omega)$ (the `chi` helper, $\omega_0 = 1$, $\gamma = 0.3$) and locate its poles with
   `numpy.roots`.
2. Confirm both poles have $\operatorname{Im}\omega = -\gamma/2 < 0$, and conclude $\chi$ is
   analytic in the upper half-plane.
3. Show the time-domain response $\chi(t) = \theta(t)\,e^{-\gamma t/2}\sin(\Omega t)/\Omega$
   vanishes for $t < 0$, Fourier-transform it numerically ($\int_0^T \chi(t)e^{i\omega t}dt$ with
   `numpy.trapezoid`), and confirm it reproduces $\chi(\omega)$ on the real axis *and* at a
   complex $\omega$ in the upper half-plane, where $e^{i\omega t}$ improves the convergence.
4. Verify the reality condition $\chi(-\omega) = \chi(\omega)^*$ numerically on a frequency grid —
   the symmetry the Kramers–Kronig integrals will use.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    pole_ims,
    [-GAMMA / 2, -GAMMA / 2],
    "the causal susceptibility's poles lie in the lower half-plane, at Im ω = −γ/2",
    rtol=1e-9,
)
validate.check(
    ft_err_max < 1e-4,
    "the t>0 impulse response Fourier-transforms back onto χ(ω), on and above the real axis",
    f"worst |FT − χ| = {ft_err_max:.1e}",
)
validate.check(
    reality_err < 1e-12,
    "the reality condition χ(−ω) = conj χ(ω) holds (Re χ even, Im χ odd)",
    f"max deviation {reality_err:.1e}",
)

## Exercise 3 — Kramers–Kronig, verified

Absorption determines dispersion — to six digits, on a function whose dispersion we can check.
Cite {eq}`eq-kramers-kronig`, {eq}`eq-sokhotski-plemelj`.

1. Derive the Kramers–Kronig relations from the Cauchy integral formula on a real-axis contour
   with a semicircular detour at $\omega$, Sokhotski–Plemelj supplying the boundary term.
2. Implement the reconstruction with the `kk_real_from_imag` helper (the folded, ω′>0 form; the
   principal value via `scipy.integrate.quad(weight='cauchy', wvar=ω)`), and reconstruct
   $\operatorname{Re}\chi$ from $\operatorname{Im}\chi$ alone at $\omega = 0.5, 1.0, 1.5$.
3. Compare to the exact $\operatorname{Re}\chi$ and confirm agreement to $\sim 10^{-6}$; plot the
   full reconstruction over a frequency band against the exact curve.
4. Verify the f-sum rule $\int_0^\infty \omega \operatorname{Im}\chi(\omega)\,d\omega = \pi/2$
   with plain `scipy.integrate.quad`, and state its meaning (a model-independent constraint tying
   the absorption spectrum to inertia).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    kk_vals,
    exact_vals,
    "Kramers–Kronig reconstructs dispersion from absorption alone",
    rtol=1e-5,
)
validate.check(
    kk_err_max < 5e-6,
    "the reconstruction reaches the promised ~1e-6 accuracy at all test frequencies",
    f"worst error {kk_err_max:.1e}",
)
validate.close(
    f_sum,
    np.pi / 2,
    "the f-sum rule: ∫ ω Im χ dω = π/2 for unit mass, independent of ω0 and γ",
    rtol=1e-6,
)

## Exercise 4 — Summation by residues: the Basel problem

The residue theorem eats an infinite series: a function with unit-residue poles at every integer
turns $\sum_n f(n)$ into contour arithmetic. Cite {eq}`eq-residue-summation`.

1. Show $\pi\cot(\pi z)$ has simple poles at every integer with residue $1$ (Laurent expansion at
   $z = n$), and confirm numerically via the limit $(z - n)\,\pi\cot(\pi z)$.
2. Derive $\sum_{n\ge1} 1/n^2 = \pi^2/6$ by integrating $\pi\cot(\pi z)/z^2$ over a growing square
   contour (`numpy.trapezoid` on the four edges, half-integer sizes so the contour threads between
   poles) and collecting the residue at $0$; watch the contour integral vanish as the square grows.
3. Confirm against the partial sums (with the integral-estimate tail $1/N$) and against
   `scipy.special.zeta`.
4. Derive and verify the shifted identity $\sum_{n\ge1} 1/(n^2+a^2) = \big(\pi\coth(\pi a)/a -
   1/a^2\big)/2$ — the warm-up whose thermal cousin is the next exercise.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    kernel_res,
    [1.0, 1.0, 1.0],
    "π cot(πz) has unit-residue simple poles at the integers — the summation kernel",
    rtol=1e-5,
)
validate.close(
    loop_over_2pii,
    2 * S_20 - np.pi**2 / 3,
    "the residue theorem on the square: ∮/(2πi) equals the enclosed residues at finite L",
    rtol=1e-4,
)
validate.close(
    [S_tail_corrected, float(zeta(2))],
    [basel, basel],
    "ζ(2) = π^2/6: tail-corrected series and scipy.special.zeta agree with the residue result",
    rtol=1e-9,
)
validate.close(
    shifted_sum,
    shifted_closed,
    "the shifted identity Σ 1/(n^2+a^2) = (π coth(πa)/a − 1/a^2)/2 — the thermal warm-up",
    rtol=1e-5,
)

## Exercise 5 — Matsubara sums: the thermal occupations emerge

The centerpiece: the frequency sums of thermal physics close by contours, and out fall the Bose
and Fermi occupation functions — before this volume has done any statistical mechanics.
Cite {eq}`eq-matsubara`.

1. Show the Bose weight $\beta/(e^{\beta z}-1)$ has simple poles with unit residue at exactly the
   bosonic Matsubara frequencies $z = 2\pi i n/\beta$, and $-\beta/(e^{\beta z}+1)$ at the
   fermionic ones; confirm both numerically by the limit $(z - i\omega_n)\times(\text{weight})$.
2. Evaluate $(1/\beta)\sum_n 1/(\omega_n^2+\varepsilon^2)$ on the *bosonic* grid with the
   `matsubara_sum` helper (symmetric truncation, $n_{\max} = 3\times10^5$, the $1/n$ tail
   estimated explicitly), and confirm the closed form $(1/2\varepsilon)\coth(\beta\varepsilon/2)$
   to at least six digits.
3. Repeat on the *fermionic* grid and confirm $(1/2\varepsilon)\tanh(\beta\varepsilon/2)$.
4. Rewrite the two closed forms as $(1/2\varepsilon)[1+2n_B(\varepsilon)]$ and
   $(1/2\varepsilon)[1-2n_F(\varepsilon)]$, exhibit $n_B = 1/(e^{\beta\varepsilon}-1)$ and
   $n_F = 1/(e^{\beta\varepsilon}+1)$ numerically, and reflect (prose): the occupation functions
   of quantum statistics have appeared from contours alone —
   [§7.7](bose-einstein-fermi-dirac.ipynb) will reach the same functions by
   the grand canonical ensemble, and the two independent routes must agree.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    [res_bose, res_fermi],
    [1.0, 1.0],
    "the Bose and Fermi weights have unit-residue poles at exactly the Matsubara frequencies",
    rtol=1e-5,
)
validate.close(
    [bose_sum, fermi_sum],
    [(1 + 2 * n_B) / (2 * eps_gap), (1 - 2 * n_F) / (2 * eps_gap)],
    "Matsubara sums yield the Bose and Fermi occupation functions, (1/2ε)[1±2n_(B/F)]",
    rtol=1e-6,
)
validate.close(
    [bose_closed, fermi_closed],
    [(1 + 2 * n_B) / (2 * eps_gap), (1 - 2 * n_F) / (2 * eps_gap)],
    "the coth/tanh closed forms are algebraically the occupation forms (the rewrite is exact)",
    rtol=1e-12,
)

## Exercise 6 — Steepest descent and Stirling

Large $N$ concentrates integrals onto saddles — and the Stirling formula that Volume V borrowed on
credit falls out in four lines, first correction included. Cite {eq}`eq-steepest-descent`.

1. Derive the Laplace approximation $\int e^{Nf(x)}dx \approx e^{Nf(x_0)}\sqrt{2\pi/N|f''(x_0)|}$
   by expanding $f$ about its maximum, and verify it against direct `scipy.integrate.quad` for
   $f(s) = \ln s - s$ at moderate $N$ — including the $1/(12N)$ approach of the ratio to $1$.
2. Apply it to $\Gamma(N+1) = \int_0^\infty e^{N\ln t - t}\,dt$ (substitute $t = Ns$; saddle at
   $s = 1$) and obtain Stirling, $N! \approx \sqrt{2\pi N}\,(N/e)^N$.
3. Verify in log space with `scipy.special.gammaln` for $N = 50, 200, 1000$ — noting that
   `scipy.special.gamma` *overflows* beyond $N \approx 170$, the volume's log-space discipline —
   and show the log-difference from exact equals the first correction $1/(12N)$ at every $N$.
4. State the complex generalization (deform through the saddle along the steepest path) and name
   stationary phase as the oscillatory sibling — the principle WKB
   ([§6.23](../06-quantum-mechanics/wkb-semiclassical.ipynb)) was using, and the one
   the path integral's classical limit ([§7.20](imaginary-time-quantum-classical.ipynb)) will use.
   (Prose part.)

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    [ratio_errs[10], ratio_errs[30], ratio_errs[100]],
    [1 / 120, 1 / 360, 1 / 1200],
    "the Laplace approximation's error is the promised 1/(12N), measured against quad",
    rtol=5e-2,
)
validate.close(
    stirling_gaps,
    stirling_expected,
    "Stirling by steepest descent: gammaln(N+1) − ln Stirling = 1/(12N) at N = 50, 200, 1000",
    rtol=5e-2,
)

## Exercise 7 — Analytic continuation and the reflection formula

Rigidity extends functions beyond their formulas — uniquely — and one of the results is an
identity this volume has already computed without noticing. Cite {eq}`eq-continuation`.

1. State the identity theorem and explain (prose) why it makes continuation unique when it
   exists.
2. Verify $\Gamma(z)\Gamma(1-z) = \pi/\sin(\pi z)$ numerically with `scipy.special.gamma` at
   $z = 0.3$ and at a complex $z$.
3. Recognize the formula: show numerically (via `scipy.integrate.quad`, split at $x = 1$ for the
   endpoint singularity) that the keyhole integral of [§7.1](complex-analysis-residues.ipynb),
   $\int_0^\infty x^{\alpha-1}/(1+x)\,dx$, equals
   $\Gamma(\alpha)\Gamma(1-\alpha)$ — the keyhole result *was* the reflection formula.
4. Evaluate $\zeta(2)$ and $\zeta(3/2)$ with `scipy.special.zeta`, confirm $\zeta(2) = \pi^2/6$
   and check $\zeta(3/2)$ against its tail-corrected series; flag $\zeta(3/2) = 2.612\dots$ as the
   constant the Bose–Einstein condensation temperature
   ([§7.17](bose-einstein-condensation.ipynb)) is built from.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    refl_lhs,
    refl_rhs,
    "the reflection formula Γ(z)Γ(1−z) = π/sin(πz), at real and complex z",
    rtol=1e-9,
)
validate.close(
    key_vals,
    gamma_vals,
    "the keyhole integral of §7.1 equals Γ(α)Γ(1−α) — it was the reflection formula in disguise",
    rtol=1e-8,
)
validate.close(
    [zeta_2, zeta_32],
    [np.pi**2 / 6, zeta_32_series],
    "scipy.special.zeta: ζ(2) = π^2/6 recovered, ζ(3/2) = 2.612… checked against its series",
    rtol=1e-6,
)

## Exercise 8 — Dispersion from absorption data alone

The Kramers–Kronig relations as a working tool: given only a sampled absorption spectrum — a
finite grid, a finite bandwidth, perhaps noise — reconstruct the dispersion that was never
measured, and be honest about the errors. Cite {eq}`eq-kramers-kronig`.

1. Sample $\operatorname{Im}\chi(\omega)$ of the oscillator on a finite uniform grid (the
   "measured absorption spectrum", $\omega' \in [0, 6]$), and prepare a second copy with small
   Gaussian noise (`numpy.random.default_rng`, $\sigma = 10^{-3}$).
2. Reconstruct $\operatorname{Re}\chi(\omega)$ from the sampled data by a discretized
   principal-value (Hilbert-transform) sum over the folded kernel $2\omega'\operatorname{Im}
   \chi(\omega')/(\omega'^2-\omega^2)$, handling the $\omega' = \omega$ singularity by symmetric
   exclusion: evaluate at each grid point using only the *opposite-parity* sample points, so
   every retained offset comes in cancelling $\pm$ pairs and no self-term ever arises.
3. Compare to the exact dispersion on an interior band; quantify the finite-bandwidth error
   against the analytic tail estimate $\sim 2\gamma/(3\pi W^3)$, and measure the extra error the
   noisy copy produces.
4. Reflect (prose): this is how optical constants are actually completed from reflectivity or
   absorption data — with the finite-bandwidth caveat being the practitioner's daily bread.

In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.check(
    err_clean < 2e-3,
    "the discretized KK reconstruction matches the exact dispersion within the stated error budget",
    f"err {err_clean:.1e} vs bandwidth-tail scale {tail_estimate:.1e}",
)
validate.check(
    err_noisy < 10.0 * err_clean and err_noisy < 2e-2,
    "1e-3 channel noise degrades the reconstruction gracefully, not catastrophically",
    f"noisy err {err_noisy:.1e}",
)

## Exercise 9 — Theorems you can measure

This notebook's four results share one engine and one moral. Causality — a statement about time —
became a statement about the complex plane, and from it fell relations between measurable
quantities that no experiment has ever violated: the absorption spectrum of a causal medium
*contains* its refractive index, recoverable by an integral we carried out to six digits and then
again, as a practitioner would, on sampled data with its bandwidth honestly billed. Thermal sums —
objects of statistical mechanics — closed by contours, and handed us the Bose and Fermi functions
before we had defined a temperature properly: the $\pm 1$ that will organize all of quantum
statistics appeared as nothing more than the choice between two ladders of poles. Large
factorials — combinatorics — reduced to the curvature of a single saddle, finally supplying the
derivation [§5.3](../05-classical-stat-mech/large-n-limit.ipynb) deferred, with the $1/(12N)$
correction sitting exactly where the theory put it. And a
formula proved on one sliver of the plane extended, uniquely, everywhere — whereupon it turned out
we had already computed it: the keyhole integral of [§7.1](complex-analysis-residues.ipynb) *was*
the reflection formula, two
notebooks meeting in one identity.

There is something almost unfair about Kramers–Kronig in particular. Nature is not obliged to let
us compute what we did not measure — yet because light cannot outrun its cause, the absorption
spectrum quietly contains the refractive index, and vice versa, forever. Analyticity is physics'
quiet enforcer: assume only that effects follow causes and that functions have derivatives, and
the bookkeeping of the complex plane does the rest. The next notebook
([§7.3](statistical-toolkit.ipynb)) assembles the volume's
working toolkit — densities of states, the Gamma–zeta–polylogarithm family, the Bose and Fermi
integrals evaluated three ways — and then the physics proper begins.

## Notebook summary

The machinery of [§7.1](complex-analysis-residues.ipynb), pointed at physics: four applications,
each a theorem of analyticity in
working clothes.

- **Sokhotski–Plemelj** {eq}`eq-sokhotski-plemelj`: the $i\varepsilon$ prescription splits a pole
  on the path into a principal value (dispersion) and a nascent delta (absorption) — verified as
  an $\varepsilon$-sequence with its honest first-order rate, with every principal value computed
  by `scipy.integrate.quad(weight='cauchy')`.
- **Causality forces analyticity** {eq}`eq-causality`: the damped oscillator's poles sit at
  $\operatorname{Im}\omega = -\gamma/2$, its $t>0$ impulse response transforms back onto
  $\chi(\omega)$ on and above the real axis, and $\chi(-\omega) = \chi(\omega)^*$.
- **Kramers–Kronig** {eq}`eq-kramers-kronig`: dispersion reconstructed from absorption alone to
  $10^{-6}$; the f-sum rule $\int\omega\operatorname{Im}\chi\,d\omega = \pi/2$ exactly; and the
  same transform run as a practitioner would, on finite noisy data, with the bandwidth tail as
  the dominant, quantified systematic.
- **Summation by residues** {eq}`eq-residue-summation`: $\pi\cot(\pi z)$'s unit residues turn
  series into contours — Basel's $\zeta(2) = \pi^2/6$ with the square contour watched vanishing,
  and the $\coth$ identity as thermal warm-up.
- **Matsubara sums** {eq}`eq-matsubara`: the Bose and Fermi weights ladder the imaginary axis, and
  the flagship sums close to $(1/2\varepsilon)[1 + 2n_B]$ and $(1/2\varepsilon)[1 - 2n_F]$ —
  the occupation functions of quantum statistics, derived from contours seven digits deep before
  any ensemble theory; [§7.7](bose-einstein-fermi-dirac.ipynb) will reach them independently.
- **Steepest descent** {eq}`eq-steepest-descent`: large-$N$ integrals concentrate onto saddles;
  Stirling is derived (not borrowed), its $1/(12N)$ correction measured at every $N$, and the
  float overflow at $N \approx 170$ converted into the volume's log-space discipline via
  `scipy.special.gammaln`.
- **Analytic continuation** {eq}`eq-continuation`: unique by the identity theorem; the reflection
  formula verified at real and complex arguments and unmasked as the keyhole of
  [§7.1](complex-analysis-residues.ipynb); $\zeta(3/2) =
  2.612\dots$ computed and flagged for the condensation temperature of
  [§7.17](bose-einstein-condensation.ipynb).

Analyticity is not a convenience here but a consequence — of causality, of temperature's discrete
frequency ladder, of large numbers — and each consequence ended in a number a measurement could
check.

## Outlook

- **The statistical toolkit ([§7.3](statistical-toolkit.ipynb)).** Densities of states, the
  Gamma–zeta–polylogarithm family, the
  Bose and Fermi integrals — evaluated directly, by series, and by the contours of this notebook —
  and the Sommerfeld expansion.
- **The ensemble route ([§7.7](bose-einstein-fermi-dirac.ipynb)).** The grand canonical derivation
  of $n_B$ and $n_F$ from the machinery of
  [§5.9](../05-classical-stat-mech/grand-canonical-ensemble-equivalence.ipynb), meeting this
  notebook's contour derivation head-on.
- **The payoffs downstream.** $\zeta(3/2)$ and the Bose–Einstein condensation temperature
  ([§7.17](bose-einstein-condensation.ipynb));
  the path integral's saddle and the classical limit
  ([§7.20](imaginary-time-quantum-classical.ipynb)).
- **The horizons, named.** Matsubara Green's functions and the many-body formalism; the
  Kubo/linear-response framework behind Kramers–Kronig (Volume VIII).
- **Cross-reference** [§7.1](complex-analysis-residues.ipynb) (the machinery; the keyhole = the
  reflection formula), Volume I (the
  damped driven oscillator), [§6.24](../06-quantum-mechanics/time-dependent-perturbation.ipynb)
  (absorption and linewidths), [§5.3](../05-classical-stat-mech/large-n-limit.ipynb) (Stirling),
  [§6.23](../06-quantum-mechanics/wkb-semiclassical.ipynb) (WKB as
  stationary phase), [§5.9](../05-classical-stat-mech/grand-canonical-ensemble-equivalence.ipynb)
  (the ensemble machinery [§7.7](bose-einstein-fermi-dirac.ipynb) will point at the same
  occupations).

In [ ]:
from ecp.style import footer

footer()